In [3]:
from langchain_community.llms import Ollama
from langchain_community.document_loaders import PyPDFLoader
from langchain.prompts import PromptTemplate
from langchain_community.vectorstores import DocArrayInMemorySearch
from langchain_community.embeddings import OllamaEmbeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from sys import argv

# 1. Create the model
llm = Ollama(model='llama3.2')
embeddings = OllamaEmbeddings(model='llama3.2')

# 2. Load the PDF file and create a retriever to be used for providing context
loader = PyPDFLoader("/home/user/Documents/jobapp/CL.pdf")
pages = loader.load_and_split()
store = DocArrayInMemorySearch.from_documents(pages, embedding=embeddings)
retriever = store.as_retriever()

# 3. Create the prompt template
template = """
Answer the question based only on the context provided.

Context: {context}

Question: {question}
"""

prompt = PromptTemplate.from_template(template)

def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

# 4. Build the chain of operations
chain = (
  {
    'context': retriever | format_docs,
    'question': RunnablePassthrough(),
  }
  | prompt
  | llm
  | StrOutputParser()
)



In [4]:
# 5. Start asking questions and getting answers in a loop
# while True:
#   question = input('What do you want to learn from the document?\n')
#   print()
print(chain.invoke({'question': "What is the summary of this document"}))
  # print()

The document appears to be a job application letter from Paul Eger, a highly motivated software engineer, applying for the position of Senior Software Engineer. He highlights his passion for technology and architecture, as well as his experience working on large projects across various industries, showcasing his technical knowledge and skills.


In [9]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
import chromadb as chdb
from langchain_community.llms import Ollama
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langsmith import traceable
from tqdm import tqdm

In [ ]:
persist_directory = "./data/chroma_db"
model_name = "llama3.2"
pdf_path = "/home/user/Documents/jobapp/CL.pdf"


    # Check if the Chroma store already exists
if os.path.exists(persist_directory):
    vectorstore = chrom(persist_directory=persist_directory, embedding_function=OllamaEmbeddings(model=model_name))
else:
        # 1. Load the PDF
        loader = PyPDFLoader(pdf_path)
        pages = loader.load()

        # 2. Split the text into chunks
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200,
            length_function=len
        )
        splits = text_splitter.split_documents(pages)

        # 3. Create embeddings and store them
        embeddings = OllamaEmbeddings(model=model_name)
        vectorstore = Chroma(persist_directory=persist_directory, embedding_function=embeddings)

        for i, chunk in enumerate(tqdm(splits, desc="Processing chunks"), 1):
            vectorstore.add_documents([chunk], embedding=embeddings)


RuntimeError: Chroma is running in http-only client mode, and can only be run with 'chromadb.api.fastapi.FastAPI' or 'chromadb.api.async_fastapi.AsyncFastAPI' as the chroma_api_impl.             see https://docs.trychroma.com/guides#using-the-python-http-only-client for more information.